In [24]:
from langchain_community.document_loaders import   DirectoryLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
import faiss,os
from langchain_community.vectorstores import FAISS
from langchain_community.llms import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
import numpy as np
from dotenv import load_dotenv
from langchain_core.prompts import ChatPromptTemplate
#hf token 
load_dotenv()  
hf_token = os.getenv("HF_TOKEN")
# print(hf_token)
os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token

In [25]:
loader= DirectoryLoader('Final_all/documents',glob='*.pdf',loader_cls=PyPDFLoader)
# docs=loader.lazy_load()

In [26]:
#KG implementation
from dataclasses import dataclass
from typing import List, Dict

@dataclass
class Triple:
    subj: str
    rel: str
    obj: str

class SimpleKG:
    def __init__(self):
        self.triples: List[Triple] = []

    def add_triple(self, subj: str, rel: str, obj: str):
        self.triples.append(Triple(subj, rel, obj))

    def find_triples(self, entity: str) -> List[Triple]:
        # return all triples where entity is subject or object
        return [t for t in self.triples if t.subj == entity or t.obj == entity]


KG = SimpleKG()
import spacy

nlp = spacy.load("en_core_web_sm")

def extract_triples_spacy(text):
    doc = nlp(text)
    triples = []
    for token in doc:
        if token.dep_ in ("ROOT", "relcl"):  # verbs or relations
            subj = [w.text for w in token.lefts if w.dep_ in ("nsubj", "nsubjpass")]
            obj = [w.text for w in token.rights if w.dep_ in ("dobj", "pobj", "attr")]
            if subj and obj:
                triples.append((" ".join(subj), token.lemma_, " ".join(obj)))
    return triples




#link entities in query to KG entities
import spacy
nlp = spacy.load("en_core_web_sm")
def extract_entity_mentions(text):
    doc = nlp(text)
    return [ent.text for ent in doc.ents] or [chunk.text for chunk in doc.noun_chunks]

def link_entities(query, kg_entities):
    # simple substring + optional embedding similarity
    mentions = extract_entity_mentions(query)  
    entity_map = {}
    for m in mentions:
        entity_map[m] = [e for e in kg_entities if m.lower() in e.lower()]
    return entity_map

def retrieve_kg_context(query, KG: SimpleKG):
    kg_entities = list(set([t.subj for t in KG.triples] + [t.obj for t in KG.triples]))
    entity_map = link_entities(query, kg_entities)
    triples_text = []
    for ents in entity_map.values():
        for ent in ents:
            for t in KG.find_triples(ent):
                triples_text.append(f"{t.subj} {t.rel} {t.obj}")
    return "\n".join(triples_text)



In [27]:

#combine kg retrieval and context retrieval
def get_combined_context(query, retriever, KG):
    # 1. Retrieve text from FAISS DB
    text_docs = retriever.invoke(query)
    text_context = "\n\n".join([d.page_content for d in text_docs])
    #print(f"context:{text_context}")
    if(len(text_docs)==0):
        return None
    
    # 2. Retrieve KG triples
    kg_context = retrieve_kg_context(query, KG)

    # 3. Combine for final LLM input
    combined_context = f"KG Facts:\n{kg_context}\n\nTextual Context:\n{text_context}"
    return combined_context

# def get_combined_context(query, retriever, KG):
#     # 1. Text Retrieval & Deduplication
#     text_docs = retriever.invoke(query)[:3]
    
    
#     unique_texts = list(dict.fromkeys([d.page_content.strip() for d in text_docs]))
#     clean_text = "\n\n".join(unique_texts)

#     # 2. KG Retrieval & Deduplication
#     kg_raw = retrieve_kg_context(query, KG)[:100] # Assuming this returns a string or list
    
    
#     # If it's a string with newlines:
#     if isinstance(kg_raw, str):
#         kg_lines = [line.strip() for line in kg_raw.split('\n') if line.strip()]
#         unique_kg = list(dict.fromkeys(kg_lines))
#         clean_kg = "\n".join(unique_kg)
#     # If it's already a list:
#     else:
#         unique_kg = list(dict.fromkeys([str(fact).strip() for fact in kg_raw]))
#         clean_kg = "\n".join(unique_kg)

#     return f"KG Facts:\n{clean_kg}\n\nTextual Context:\n{clean_text}"

In [28]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,     
    chunk_overlap=100,   
    separators=["\n\n", "\n", " ", ""]
)

questions="Can you tell me about the return policy?" #dummy question for triple extraction
ground_truths=[]
# doc=""
q=0

docs = list(loader.lazy_load())

print("Loaded docs:", len(docs))
print("Sample length:", len(docs[0].page_content))

chunks = text_splitter.split_documents(docs)

print("Total chunks:", len(chunks))

Loaded docs: 7
Sample length: 3664
Total chunks: 33


In [29]:
from langchain_community.embeddings import HuggingFaceEmbeddings
import torch

In [30]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2",
                                   model_kwargs={"device": "cuda"},
                                   encode_kwargs={"normalize_embeddings": True}
                                   )

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4470.75it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [31]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())

2.11.0+cu130
True


In [32]:
import transformers
print(transformers.__version__)

5.5.0


In [33]:
#FAISS cosine similarity index
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

dimension = 784


index = faiss.IndexFlatIP(dimension)

In [34]:

vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local("faiss_index")
print(len(chunks))
print(vectorstore.index.ntotal)

33
33


In [35]:

retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
    )

In [36]:
import transformers
print(transformers.__version__)  # needs to be >= 4.43

5.5.0


In [ ]:
from langchain_community.llms import HuggingFacePipeline
from huggingface_hub import login
llm = HuggingFacePipeline.from_model_id(
    model_id="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    pipeline_kwargs={
        "temperature": 0.1,
        "max_new_tokens": 256,
        "do_sample": False,
    },
    device_map="auto",
)

/home/souradeep/miniconda3/envs/ie624/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
Loading weights: 100%|██████████| 291/291 [00:00<00:00, 7352.09it/s]
The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'temperature', 'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:

# You are a factual assistant whose goal is to produce answers strictly supported by the provided context.



# Instructions:
# 1. Use ONLY the information present in the provided context.
# 2. Do NOT introduce external knowledge.
# 3. If the answer cannot be found in the context, respond with No answer.
# 4. Ensure the answer directly addresses the question.
# 5. Answer in maximum 3 to 5 words
# 6. If the answer is implied, infer it.
# 7. If the answer is Rollo then give no answer at all.


# Context:
# {context}

# Question:
# {question}

# Refined Answer:
actor_prompt = PromptTemplate(
    input_variables=["context", "question","feedback_section"],
    template="""
You are a factual assistant whose goal is to produce answers strictly supported by the provided context.



Instructions:
1. Use ONLY the information present in the provided context.
2. Do NOT introduce external knowledge.
3. If the answer cannot be found in the context, don't give any answer.
4. Ensure the answer directly addresses the question.
5. Answer in maximum 3 to 5 words
6. If the answer is implied, infer it.
7. If the answer is Rollo then give no answer at all.


Context:
{context}

Question:
{question}

Refined Answer:
"""
)

def actor(question, context, feedback_section):
    actor_chain = actor_prompt | llm
    # context = get_combined_context(question,retriever, KG)
    response = actor_chain.invoke({
            "context":context,
            "feedback_section": feedback_section,
            "question":question
            }
            )
    answer=response.split('Answer:')[-1].strip()
    return answer


In [ ]:
# prompt = PromptTemplate(
#     input_variables=["context", "question"],
#     template="""
# You are a factual assistant. Use the following context to answer the question.
# Do NOT add information that is not supported by the context.
# if you don't find the answer in the context then simply say NO.

# Context:
# {context}

# Question: {question}
# Answer:
# """
# )

In [ ]:
#evaluator
import json

In [ ]:
eval_prompt = PromptTemplate(
    input_variables=["query", "context", "answer"],
    template="""You are a RAG Quality Auditor. You must evaluate ONLY two things:
    1. CONTEXT RELEVANCE: Does the retrieved context contain the information needed to answer the user's query?
    2. FAITHFULNESS: Is the provided answer derived ONLY from the retrieved context?
     
    Output ONLY a JSON object with EXACTLY this structure:
    {{
        "context_relevance": {{
            "score": float (0.0 to 1.0),
            "reason": "string"
        }},
        "faithfulness": {{
            "score": float (0.0 to 1.0),
            "reason": "string"
        }},
        "final_status": "PASS" or "FAIL",
        "action_item": "What should the system do next?(e.g., 'Re-retrieve context' or 'Refine answer').
        if context_relevance score<0.8 then 'Re-retrieve context',if faithfullness score<0.8 then 'Refine answer'.
    }}
    
    Threshold for PASS: Both scores must be >= 0.8.
     
USER QUERY: {query}

RETRIEVED CONTEXT: {context}

GENERATED ANSWER: {answer}
"""
)

evaluator_llm = llm

/tmp/ipykernel_3836651/2136529005.py:1: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  evaluator_llm= ChatOllama(model="qwen2.5:7b",


In [ ]:
def evaluate_response(question,context, answer):
    chain = eval_prompt | evaluator_llm
    # response = chain.invoke({"context": context, "answer": answer})
    
    
    try:
        response = chain.invoke({
            "query": question,
            "context": context,
            "answer": answer
        })
        return json.loads(response.content)
    except Exception as e:
        return {
            "final_status": "FAIL",
            "action_item": f"Error in evaluation: {str(e)}"
        }

In [ ]:
try:
    response = evaluator_llm.invoke("Hello!")
    print("Local LLM is responding!")
except Exception as e:
    print(f"Connection failed. Is the model loaded? Error: {e}")

Connection failed. Is the server running? Error: HTTPConnectionPool(host='localhost', port=11434): Max retries exceeded with url: /api/chat (Caused by NewConnectionError("HTTPConnection(host='localhost', port=11434): Failed to establish a new connection: [Errno 111] Connection refused"))


In [ ]:
query_refiner_llm = llm

In [ ]:
def run_reflexion_loop(original_question,max_iterations=3):
    
    search_query = original_question
    current_context = get_combined_context(search_query, retriever, KG)
    feedback_section = " " 
    
    for i in range(max_iterations):
        # print(f"\n--- Iteration {i+1} ---")
        
        answer = actor(original_question, current_context, feedback_section)
        
        evaluation = evaluate_response(original_question, current_context, answer)
        # print(json.dumps(evaluation, indent=2))
        # print(current_context)
        # print(f"answer:{answer}")

        if evaluation.get('final_status') == "PASS":
            return answer,current_context
        
        action = evaluation.get('action_item', 'Refine answer')
        
        if action == "Re-retrieve context":
            search_query = generate_refined_query(original_question, evaluation['context_relevance']['reason'])
            # print(f"new search query:{search_query}")
            current_context = get_combined_context(search_query, retriever, KG)
        else:
            feedback_section=evaluation['faithfulness']['reason']
        # feedback_section = f"""
        # ### FEEDBACK FROM PREVIOUS ATTEMPT ###
        # - Context Relevance Score: {evaluation['context_relevance']['score']}
        # - Context Critique: {evaluation['context_relevance']['reason']}
        
        # - Faithfulness Score: {evaluation['faithfulness']['score']}
        # - Faithfulness Critique: {evaluation['faithfulness']['reason']}
        
        # - Required Action: {evaluation['action_item']}
        # ######################################
        # """
        
        # print(f"Critique sent to Actor: {evaluation['action_item']}")
        
    # print(" Max iterations reached.")
    return answer,current_context

In [ ]:
question=qa_pairs[93]["question"]
print(question)
answer,context=run_reflexion_loop(question)
print(context)
print(f"answer:{answer}")
print(qa_pairs[93]["answer"])

What kind of force did Harthacnut establish?


/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:567: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.1` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/mnt/nas/shuvranshu/.conda/envs/ragenv/lib/python3.10/site-packages/transformers/generation/configuration_utils.py:572: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.9` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(


KG Facts:


Textual Context:
Some Normans joined Turkish forces to aid in the destruction of the Armenians vassal-states of Sassoun and Taron in far eastern Anatolia. Later, many took up service with the Armenian state further south in Cilicia and the Taurus Mountains. A Norman named Oursel led a force of "Franks" into the upper Euphrates valley in northern Syria. From 1073 to 1074, 8,000 of the 20,000 troops of the Armenian general Philaretus Brachamius were Normans—formerly of Oursel—led by Raimbaud. They even lent their ethnicity to the name of their castle: Afranji, meaning "Franks." The known trade between Amalfi and Antioch and between Bari and Tarsus may be related to the presence of Italo-Normans in those cities while Amalfi and Bari were under Norman rule in Italy.
answer:
N/A


In [ ]:
# question="Explain the legal basis and the override power of the inserted rule 31D."
# answer=run_reflexion_loop(question)
# print(f"answer:{answer}")

results = []
rag_answers=[]
retrieved_contexts=[]
q=0

# for i, qa in enumerate(qa_pairs):
#     question = qa["question"]
#     gt = qa["answer"]
#     # response,context = run_reflexion_loop(question)
#     context = get_combined_context(question, retriever, KG)
#     response = actor(question, context, feedback_section=" ")
#     results.append({
#         "question": question,
#         "pred": response,
#         "gt": gt,
#         "context":context
#     })

#     print(f"q{i}:{response}")
for question in questions:
    answer,context = run_reflexion_loop(question)
    retrieved_contexts.append(context)
    # response = actor(question, context, feedback_section=" ")
    ans=answer.split('Answer:')[-1].strip()
    rag_answers.append(ans)
    print(f"qa {q}:{ans}")
    q+=1

Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


qa 0:France
qa 1:under Richard I of Normandy
qa 2:Denmark, Iceland and Norway
qa 3:Rollo


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


qa 4:first half of the 10th century
qa 5:Normans
qa 6:no answer
qa 7:no answer
qa 8:no answer
qa 9:Duke William II of Normandy
qa 10:Rollo
qa 11:Christian piety
qa 12:no answer
qa 13:No answer
qa 14:The descendants of Rollo's Vikings
qa 15:Rollo
qa 16:no answer
qa 17:Norseman
qa 18:9th century
qa 19:Norseman
qa 20:French be people
qa 21:911
qa 22:King Charles III of West Francia
qa 23:Epte
qa 24:no answer
qa 25:no answer
qa 26:Rollo
qa 27:treaty offer Rollo lands
qa 28:Norman culture
qa 29:880s
qa 30:Danes, Norwegians, Norse–Gaels, Orkney Vikings, possibly Swedes, and Anglo-Danes from the English Danelaw under Norse control
qa 31:Christian piety
qa 32:Normandy
qa 33:Catholicism (Christianity)
qa 34:Frankish heritage
qa 35:no answer
qa 36:musical traditions
qa 37:Crispin lead Normans
qa 38:culture
qa 39:the Anglo-Norman king Richard the Lion-Heart
qa 40:Seljuk Turks
qa 41:Normans
qa 42:Pechenegs, the Bulgars, and especially the Seljuk Turks
qa 43:no answer
qa 44:Sicilian campaign of Geo

In [ ]:
#EM
# correct = sum([pred.strip().lower() == gt.strip().lower()
#                for pred, gt in zip(rag_answers, ground_truths)])

# total = len(ground_truths)

# print("Correct:", correct)
# print("Accuracy:", correct / total)
def normalize(text):
    text = text.strip().lower()
    if text == "no answer":
        return ""
    return text

correct = sum([
    normalize(pred) == normalize(gt)
    for pred, gt in zip(rag_answers, ground_truths)
])

total = len(ground_truths)

print("Correct:", correct)
print("Accuracy:", correct / total)

Correct: 41
Accuracy: 0.41


In [ ]:
#substring match
correct = sum([gt.lower() in pred.lower()
               for pred, gt in zip(rag_answers, ground_truths)])

print("Accuracy:", correct / len(ground_truths))

# def normalize(text):
#     text = text.strip().lower()
#     if text in ["no answers", "no answer", "no_answer", "n/a"]:
#         return ""
#     return text

# correct = sum([
#     normalize(gt) in normalize(pred)
#     for pred, gt in zip(rag_answers, ground_truths)
# ])

# print("Accuracy:", correct / len(ground_truths))

Accuracy: 0.83


In [ ]:
#exact match and F1 score
import re
import string

def normalize_answer(s):
    def remove_articles(text):
        return re.sub(r'\b(a|an|the)\b', ' ', text)

    def white_space_fix(text):
        return ' '.join(text.split())

    def remove_punc(text):
        return ''.join(ch for ch in text if ch not in set(string.punctuation))

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))

In [ ]:
def compute_em(pred, gt):
    return int(normalize_answer(pred) == normalize_answer(gt))

In [ ]:
def compute_f1(pred, gt):
    pred_tokens = normalize_answer(pred).split()
    gt_tokens = normalize_answer(gt).split()

    common = set(pred_tokens) & set(gt_tokens)
    if len(common) == 0:
        return 0.0

    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gt_tokens)

    return 2 * precision * recall / (precision + recall)

In [ ]:
em_scores = []
f1_scores = []

for pred, gt in zip(rag_answers, ground_truths):
    

    if pred is None or pred.strip() == "":
        continue   # skip bad outputs

    em_scores.append(compute_em(pred, gt))
    f1_scores.append(compute_f1(pred, gt))

avg_em = sum(em_scores) / len(em_scores)
avg_f1 = sum(f1_scores) / len(f1_scores)

print("Exact Match (EM):", avg_em)
print("F1 Score:", avg_f1)

Exact Match (EM): 0.24
F1 Score: 0.275952380952381


In [ ]:

#ragas evaluation
# dataset_ragas = []

# for i, item in enumerate(results):
#     dataset_ragas.append({
#         "user_input": item["question"],
#         "retrieved_contexts": item["context"] if isinstance(item["context"], list) else [item["context"]],
#         "response": item["pred"],
#         "reference": item["gt"]  # ✅ correct GT
#     })
#ragas evaluation
dataset_ragas = []

for q, ctx, ans, ref in zip(questions, retrieved_contexts, rag_answers, ground_truths):
    dataset_ragas.append({
        "user_input": q,
        "retrieved_contexts": ctx if isinstance(ctx, list) else [ctx],
        "response": ans,
        "reference": ref
    })

In [ ]:
from ragas import EvaluationDataset
evaluation_dataset = EvaluationDataset.from_list(dataset_ragas)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from ragas import evaluate
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.metrics import LLMContextRecall, Faithfulness, FactualCorrectness
# answer_relevancy, context_precision

In [ ]:
json_prompt = """
You are a strict JSON output generator used for evaluation in RAG systems.
You must always reply ONLY with a valid JSON object — no explanations, no extra text.

If the question asks for a score, output: {"score": <float between 0 and 1>}
If the question asks for a label (e.g., yes/no), output: {"label": "yes"} or {"label": "no"}
Do not include extra keys, comments, or markdown fences.
"""
langchain_llm = llm


langchain_embeddings=HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

/tmp/ipykernel_342809/241535895.py:12: LangChainDeprecationWarning: The class `OllamaEmbeddings` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import OllamaEmbeddings``.
  langchain_embeddings=OllamaEmbeddings(model="nomic-embed-text")


In [ ]:
result = evaluate(dataset=evaluation_dataset,
                  batch_size=2,
                #   metrics=[ LLMContextRecall(), Faithfulness(),FactualCorrectness()],
                  metrics=[LLMContextRecall(), Faithfulness(), FactualCorrectness()],
                  llm=langchain_llm,embeddings=langchain_embeddings)
print(result)

Evaluating:   1%|          | 2/300 [00:09<22:35,  4.55s/it]Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt context_recall_classification_prompt failed to parse output: The output parser failed to parse the output including retries.
Exception raised in Job[3]: RagasOutputParserException(The output parser failed to parse the output including retries.)
Evaluating:   1%|▏         | 4/300 [00:19<23:36,  4.79s/it]Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parser failed to parse the output including retries.
Prompt fix_output_format failed to parse output: The output parse

{'context_recall': 0.5533, 'faithfulness': 0.6749, 'factual_correctness(mode=f1)': 0.3945}


In [ ]:
from sentence_transformers import SentenceTransformer, util
import torch
import numpy as np

qa_model = SentenceTransformer("multi-qa-MiniLM-L6-cos-v1", device="cuda:0")


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 835.42it/s]
BertModel LOAD REPORT from: sentence-transformers/multi-qa-MiniLM-L6-cos-v1
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:

scores = []
skipped = 0
for row in evaluation_dataset:
    if not row.reference or row.reference.strip() == "":  # skip empty references
        skipped += 1
        continue
    
    ref_emb = qa_model.encode(row.reference, convert_to_tensor=True)
    ans_emb = qa_model.encode(row.response, convert_to_tensor=True)
    score = util.cos_sim(ref_emb, ans_emb).item()
    scores.append(score)

print(f"Skipped {skipped} samples with no reference")
print(f"Valid samples: {len(scores)}/{len(evaluation_dataset)}")
print(f"answer_similarity (generated vs reference): {np.mean(scores):.4f}")

# def normalize(text):
#     if not text:
#         return ""
#     text = text.strip().lower()
#     if text in ["no answers", "no answer", "no_answer", "n/a"]:
#         return ""
#     return text

# scores = []
# skipped = 0

# for row in evaluation_dataset:
#     # normalize both reference and response
#     reference = normalize(row.reference)
#     response = normalize(row.response)

#     # skip if reference is empty
#     if reference == "":
#         skipped += 1
#         continue

#     # handle case where response becomes empty
#     if response == "":
#         score = 0.0   # no answer → worst similarity
#     else:
#         ref_emb = qa_model.encode(reference, convert_to_tensor=True)
#         ans_emb = qa_model.encode(response, convert_to_tensor=True)
#         score = util.cos_sim(ref_emb, ans_emb).item()

#     scores.append(score)

# print(f"Skipped {skipped} samples with no reference")
# print(f"Valid samples: {len(scores)}/{len(evaluation_dataset)}")
# print(f"answer_similarity (generated vs reference): {np.mean(scores):.4f}")

Skipped 55 samples with no reference
Valid samples: 45/100
answer_similarity (generated vs reference): 0.7118
